# E6 on Kaggle: Tent continual with a periodic reset

Runs `scripts/run_e6.sh` on the two models where continual Tent drifts (Standard seed 2, CutMix seed 1):
reset every 30, 100 or 320 batches, 3 runs each (about 50 minutes on a T4).

Same setup as `stage2_kaggle.ipynb`: **GPU T4 x2**, **Internet On**, **Add Input** → the Stage 1 models dataset,
then **Save Version → Save & Run All (Commit)**, and download `stage2_results.zip` from the **Output** tab.

In [ ]:
BRANCH = 'member2/stage2-adaptation-eval'   # change to 'main' after the merge
HF_REPO = 'YashP54/cifar10-c'               # Hugging Face mirror of CIFAR-10-C

!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /tmp
!rm -rf repo && git clone -q -b $BRANCH https://github.com/KirtanPatel30/Augmentation-Test-time-Adaptation.git repo
%cd /tmp/repo
!git log --oneline -1
# Data stays in /tmp (not saved). Results are written straight to /kaggle/working/results,
# which Kaggle saves as the notebook's output. (Not a symlink: the repo already has a results/ folder.)
OUT = '/kaggle/working/results'
!mkdir -p /tmp/data $OUT && ln -sfn /tmp/data data

In [ ]:
!CIFAR10C_HF_REPO=$HF_REPO bash scripts/download_cifar10c.sh data
!python -m pytest -q tests

In [ ]:
import glob, os
MODELS = ['standard_seed2', 'cutmix_seed1']
CKPTS = sorted(c for c in glob.glob('/kaggle/input/**/final.pt', recursive=True)
               if os.path.basename(os.path.dirname(c)) in MODELS)
print(len(CKPTS), 'models:', *CKPTS, sep='\n  ')
assert len(CKPTS) == len(MODELS), 'Missing a model: add the Stage 1 models dataset with Add Input'

In [ ]:
# About 25 minutes per model on a T4. The log is also saved to the output.
import subprocess
def run_stage2(ckpts, **env):
    if not ckpts:
        return
    with open(f'{OUT}/run_stage2.log', 'a') as log:
        p = subprocess.Popen(['bash', 'scripts/run_e6.sh', *ckpts], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, env={**os.environ, 'OUT': OUT, **env})
        for line in p.stdout:
            log.write(line); log.flush()
            if 'Warning' not in line and 'pkg_resources' not in line:
                print(line, end='')
    print('exit code', p.wait())

run_stage2(CKPTS)

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
pd.read_csv(f'{OUT}/tables/runs.csv')

In [ ]:
!cd /kaggle/working && zip -qr stage2_results.zip results && ls -lh stage2_results.zip && unzip -l stage2_results.zip | tail -n 1